# Bayesian Inference Demo

This notebook demonstrates Bayesian modeling capabilities using the `datascienceutils` library.

## Features Demonstrated:
- MCMC Sampling (Metropolis-Hastings)
- Bayesian A/B Testing
- Convergence Diagnostics (R-hat, ESS)
- Posterior Distributions
- Credible Intervals

In [ ]:
:dep datascienceutils-core = { path = "../datascienceutils-core", features = ["bayesian-inference"] }
:dep ndarray = "0.15"
:dep plotters = { version = "0.3", default_features = false, features = ["evcxr", "all_series"] }
:dep rand = "0.8"

In [ ]:
use datascienceutils_core::analyze::bayesian::*;
use ndarray::{array, Array1, Array2};
use plotters::prelude::*;

## 1. MCMC Sampling - Normal Distribution

Sample from a standard normal distribution using Metropolis-Hastings.

In [ ]:
// Define log posterior for N(0, 1)
let log_posterior = |x: &Array1<f64>| {
    -0.5 * x[0] * x[0]  // Log of exp(-x^2/2)
};

let initial = array![0.0];
let sampler = MCMCSampler::MetropolisHastings { proposal_std: 1.0 };
let samples = mcmc_sample(log_posterior, initial, sampler, 5000).unwrap();

println!("=== MCMC Sampling Results ===");
println!("Number of samples: {}", samples.nrows());
println!("Sample mean: {:.4} (expected: 0.0)", samples.mean().unwrap());
println!("Sample std: {:.4} (expected: 1.0)", 
    samples.iter().map(|&x| x * x).sum::<f64>().sqrt() / (samples.len() as f64).sqrt());

## 2. Convergence Diagnostics

Check MCMC convergence using R-hat and Effective Sample Size.

In [ ]:
let rhat = compute_rhat(&samples);
let ess = effective_sample_size(&samples);

println!("\n=== Convergence Diagnostics ===");
println!("R-hat: {:.4} (should be < 1.1)", rhat);
println!("Effective Sample Size: {:.0}", ess);
println!("ESS ratio: {:.2}%", (ess / samples.nrows() as f64) * 100.0);

if rhat < 1.1 {
    println!("✓ Chain has converged!");
} else {
    println!("⚠ Chain may not have converged");
}

## 3. Bayesian A/B Testing - E-commerce Example

Compare conversion rates between two website designs.

In [ ]:
// Control: Current website (10% conversion)
let control = vec![0.0; 900];
let mut control = control;
control.extend(vec![1.0; 100]);

// Treatment: New design (12% conversion)
let treatment = vec![0.0; 880];
let mut treatment = treatment;
treatment.extend(vec![1.0; 120]);

println!("=== A/B Test Setup ===");
println!("Control group: {} visitors, {} conversions ({:.1}%)", 
    control.len(), 
    control.iter().filter(|&&x| x > 0.5).count(),
    (control.iter().filter(|&&x| x > 0.5).count() as f64 / control.len() as f64) * 100.0);
println!("Treatment group: {} visitors, {} conversions ({:.1}%)", 
    treatment.len(), 
    treatment.iter().filter(|&&x| x > 0.5).count(),
    (treatment.iter().filter(|&&x| x > 0.5).count() as f64 / treatment.len() as f64) * 100.0);

In [ ]:
let result = bayesian_ab_test(&control, &treatment).unwrap();

println!("\n=== Bayesian A/B Test Results ===");
println!("Probability treatment is better: {:.1}%", result.prob_treatment_better * 100.0);
println!("Expected lift: {:.2}% points", result.expected_lift * 100.0);
println!("95% Credible Interval: [{:.2}%, {:.2}%]", 
    result.credible_interval.0 * 100.0,
    result.credible_interval.1 * 100.0);

if result.prob_treatment_better > 0.95 {
    println!("\n✓ Strong evidence for treatment! Recommend deploying new design.");
} else if result.prob_treatment_better > 0.80 {
    println!("\n⚠ Moderate evidence for treatment. Consider more data.");
} else {
    println!("\n✗ Insufficient evidence. Keep current design.");
}

## 4. Medical Trial Example

Bayesian analysis of a medical treatment trial.

In [ ]:
// Placebo: 15% recovery rate
let placebo = vec![0.0; 85];
let mut placebo = placebo;
placebo.extend(vec![1.0; 15]);

// Drug: 25% recovery rate
let drug = vec![0.0; 75];
let mut drug = drug;
drug.extend(vec![1.0; 25]);

let medical_result = bayesian_ab_test(&placebo, &drug).unwrap();

println!("=== Medical Trial Results ===");
println!("Placebo recovery rate: {:.1}%", 
    (placebo.iter().filter(|&&x| x > 0.5).count() as f64 / placebo.len() as f64) * 100.0);
println!("Drug recovery rate: {:.1}%", 
    (drug.iter().filter(|&&x| x > 0.5).count() as f64 / drug.len() as f64) * 100.0);
println!("\nProbability drug is better: {:.1}%", medical_result.prob_treatment_better * 100.0);
println!("Expected improvement: {:.1}% points", medical_result.expected_lift * 100.0);
println!("95% Credible Interval: [{:.1}%, {:.1}%]", 
    medical_result.credible_interval.0 * 100.0,
    medical_result.credible_interval.1 * 100.0);

## 5. Bayesian Linear Regression Setup

Initialize a Bayesian linear regression model.

In [ ]:
let model = BayesianLinearRegression::new(2);
println!("=== Bayesian Linear Regression ===");
println!("Model initialized with {} features", model.prior_mean.len() - 1);
println!("Prior mean: {:?}", model.prior_mean);
println!("Prior covariance shape: {:?}", model.prior_cov.dim());

## Summary

This notebook demonstrated:

1. **MCMC Sampling** - Metropolis-Hastings for posterior inference
2. **Convergence Diagnostics** - R-hat and ESS for quality checks
3. **Bayesian A/B Testing** - E-commerce conversion optimization
4. **Medical Trials** - Treatment efficacy analysis
5. **Bayesian Regression** - Model initialization

### Key Advantages of Bayesian Methods:
- **Probabilistic Statements**: "95% probability treatment is better"
- **Credible Intervals**: Natural interpretation of uncertainty
- **Small Sample Sizes**: Works well with limited data
- **Prior Knowledge**: Can incorporate domain expertise